In [1]:
import pandas as pd

    MIDAS2: GTDB R202
    VMGC: GTDB R214.1

### Download files from: https://github.com/shenwei356/gtdb-taxdump?tab=readme-ov-file

    wget https://github.com/shenwei356/gtdb-taxdump/releases/download/v0.5.0/gtdb-taxid-changelog.csv.gz
    wget https://github.com/shenwei356/gtdb-taxdump/releases/download/v0.5.0/gtdb-taxdump.tar.gz

In [2]:
base_dir = '/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/code/'

In [5]:
vmgc_sgbs = pd.read_csv(f'{base_dir}/VMGC/VMGC_db_build/VMGC_orig_files/VMGC_prokaryote_SGB.info', sep='\t', index_col=0)
genome_to_sgb = pd.read_csv(f'{base_dir}/VMGC/VMGC_db_build/genomes_and_SGBs.csv')
genome_to_sgb = genome_to_sgb[['SGB','species']].drop_duplicates().set_index('SGB').to_dict()['species']
vmgc_sgbs['VMGC_species_id'] = vmgc_sgbs.index.map(genome_to_sgb)

vmgc_sgbs = vmgc_sgbs.rename(columns={
                          'Species':'VMGC_species'})
vmgc_sgbs['VMGC_taxonomy'] = vmgc_sgbs[['Kingdom', 'Phylum', 'Class',
       'Order', 'Family', 'Genus','VMGC_species']].agg(';'.join, axis=1)

vmgc_sgbs['VMGC_species'] = vmgc_sgbs['VMGC_species'].str.replace('\t','')
vmgc_sgbs.head()

,Representative_genome_ID,Number_of_genomes,Kingdom,Phylum,Class,Order,Family,Genus,VMGC_species,Taxonomic_level,Cultured,VMGC_species_id,VMGC_taxonomy
SGB_ID,,,,,,,,,,,,,
SGB001,SRR17284223.mbin.1,1544,Bacteria,Bacillota,Bacilli,Lactobacillales,Lactobacillaceae,Lactobacillus,Lactobacillus iners,Species,Cultured (women vagina),240891,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...
SGB002,GCF_001562845.1,764,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Atopobiaceae,Fannyhessea,Fannyhessea vaginae,Species,Cultured (women vagina),619501,Bacteria;Actinomycetota;Coriobacteriia;Corioba...
SGB003,GCF_001553395.1,664,Bacteria,Bacillota_C,Negativicutes,Veillonellales,Megasphaeraceae,Megasphaera,Megasphaera lornae,Species,Cultured (women vagina),829633,Bacteria;Bacillota_C;Negativicutes;Veillonella...
SGB004,ERR10897722.mbin.1,586,Bacteria,Actinomycetota,Actinomycetia,Actinomycetales,Bifidobacteriaceae,Bifidobacterium,Bifidobacterium vaginale,Species,Cultured (women vagina),783244,Bacteria;Actinomycetota;Actinomycetia;Actinomy...
SGB005,MG1112.mbin.9,562,Bacteria,Actinomycetota,Actinomycetia,Actinomycetales,Bifidobacteriaceae,Bifidobacterium,Bifidobacterium swidsinskii,Species,Cultured (women vagina),367459,Bacteria;Actinomycetota;Actinomycetia;Actinomy...


In [6]:
def get_taxid(species, version='R214'):
    
    c = !echo {species}  | taxonkit name2taxid --data-dir gtdb-taxdump/{version}/ 

    try:
        taxid = c[0].split('\t')[1]
        if len(taxid) == 0:
            print(c)
            return
        
    except IndexError:
        print('Index error: ', species, c)
        return
    
    return taxid

vmgc_sgbs['taxid_R214'] = vmgc_sgbs['VMGC_species'].apply(get_taxid, version='R214')


Index error:  Amygdalobacter indicium (BVAB2) ['zsh:1: unknown file attribute: B']
Index error:  UBA629 sp005465875 (BVAB1) ['zsh:1: unknown file attribute: B']
['Prevotella_unknown\t']
['Fannyhessea_unknown\t']
Index error:  Mageeibacillus indolicus (BVAB3) ['zsh:1: unknown file attribute: B']
['Prevotella_unknown\t']
['Bifidobacterium_unknown\t']
['Berryella_unknown\t']
['Cryptobacteroides_unknown\t']
['Prevotella_unknown\t']
['Bifidobacterium_unknown\t']
['Bifidobacterium_unknown\t']
['Sodaliphilus_unknown\t']
['Fannyhessea_unknown\t']
['Nanoperiomorbus_unknown\t']
['Porphyromonas_unknown\t']
['Amygdalobacter nucleatus\t']
['Bifidobacterium_unknown\t']
['Fannyhessea_unknown\t']
['ZJ293_unknown\t']
['Bulleidia_unknown\t']
['Peptoniphilus_B_unknown\t']
['Atopobacter_unknown\t']
['Nanoperiomorbus_unknown\t']
['Bifidobacterium_unknown\t']
['Catonella_unknown\t']
['Eubacterium_I_unknown\t']
['Mobiluncus_unknown\t']
['Nanoperiomorbus_unknown\t']
['Corynebacterium_unknown\t']
['Peptoniphil

In [9]:
df = vmgc_sgbs[['taxid_R214','VMGC_taxonomy','VMGC_species']]
#drop species not in both versions
df = df[~df['taxid_R214'].isna()] 
df.shape

(616, 3)

In [10]:
tax_list = df['taxid_R214'].unique().tolist()

### Get taxonomic history of species across GTDB versions

In [11]:
a = pd.read_csv('gtdb-taxid-changelog.csv.gz')
all_versions = a['version'].sort_values().unique().tolist()

In [12]:
def get_lineage_by_version(t_df, v):
    
    version_hist = t_df[t_df['version'] == v]
    
    changes = version_hist['change'].tolist()
    if 'MERGE' in changes:
        change_val = version_hist['change-value'].unique().tolist()
        if len(change_val) > 1:
            print()
        else:
            change_val = change_val[0]
            l = a[(a['taxid']==int(change_val)) & (a['change'] == 'ABSORB') 
                  & (a['version'] == v)]['lineage'].unique().tolist()

    elif 'DELETE' in changes:
        l = None
        
    elif 'NEW' in changes or 'CHANGE_LIN_TAX' in changes or 'ABSORB' in changes or 'REUSE_DEL' in changes or 'REUSE_MER' in changes:
        l = version_hist['lineage'].unique().tolist()            

    if l and len(l) > 1:
        print(l)
    elif l:
        l = l[0]
    
    return l

In [13]:
def get_taxon_lineage(t, versions=['R202', 'R207', 'R214']):
    
    t_df = a[a['taxid']==t].sort_values('version')
    
    version_lineages = []    

    for v_check in versions:
        
        l = None
        v_idx = all_versions.index(v_check)
        this_version_and_before = all_versions[:v_idx+1][::-1]

        for v in this_version_and_before:
            if v in t_df['version'].tolist():
                l = get_lineage_by_version(t_df, v)
                break

            else:
                continue

        
        version_lineages += [l]

    return version_lineages

In [14]:
tax_lineages = {}

for t in tax_list:
    tax_lineages[t] = get_taxon_lineage(int(t))

In [15]:
lineage_df = pd.DataFrame.from_dict(tax_lineages, orient='index', columns=['R202', 'R207', 'R214'])
lineage_df

,R202,R207,R214
360001829,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...
1518558221,Bacteria;Actinobacteriota;Coriobacteriia;Corio...,Bacteria;Actinobacteriota;Coriobacteriia;Corio...,Bacteria;Actinomycetota;Coriobacteriia;Corioba...
459781732,None,Bacteria;Firmicutes_C;Negativicutes;Veillonell...,Bacteria;Bacillota_C;Negativicutes;Veillonella...
697922065,Bacteria;Actinobacteriota;Actinomycetia;Actino...,Bacteria;Actinobacteriota;Actinomycetia;Actino...,Bacteria;Actinomycetota;Actinomycetia;Actinomy...
8006882,None,Bacteria;Actinobacteriota;Actinomycetia;Actino...,Bacteria;Actinomycetota;Actinomycetia;Actinomy...
...,...,...,...
748464734,Bacteria;Actinobacteriota;Actinomycetia;Actino...,Bacteria;Actinobacteriota;Actinomycetia;Actino...,Bacteria;Actinomycetota;Actinomycetia;Actinomy...
1320179701,None,None,Bacteria;Actinomycetota;Actinomycetia;Actinomy...
2032408915,Bacteria;Firmicutes_A;Clostridia;Saccharoferme...,Bacteria;Firmicutes_A;Clostridia;Saccharoferme...,Bacteria;Bacillota_A;Clostridia;Saccharofermen...
1898184418,None,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...


In [16]:
merged = lineage_df.merge(df.reset_index(), left_index=True, right_on='taxid_R214')
merged = merged.rename(columns={'SGB_ID':'VMGC_SGB'})
merged.head()

,R202,R207,R214,VMGC_SGB,taxid_R214,VMGC_taxonomy,VMGC_species
0,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB001,360001829,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,Lactobacillus iners
14,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB019,360001829,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,Lactobacillus iners
112,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB135,360001829,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,Lactobacillus iners
156,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB191,360001829,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,Lactobacillus iners
182,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB219,360001829,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,Lactobacillus iners


In [17]:
merged[merged['VMGC_taxonomy'] != merged['R214']]

,R202,R207,R214,VMGC_SGB,taxid_R214,VMGC_taxonomy,VMGC_species


In [18]:
merged = merged.drop(columns='VMGC_taxonomy')

### Add/remove clade delineations for all versions

Since some programs use the names with g__, s__, etc. and some don't

In [19]:
def add_clades_to_gtdb_name(gtdb):
    
    if gtdb == None or ';' not in gtdb:
        return gtdb
    new_str = ''
    for pre,post in zip('dpcofgs', gtdb.split(';')):
        new_str += f'{pre}__{post};'
    return new_str

for col in ['R202', 'R207','R214']:
    
    merged[col+'_with_clades'] = merged[col].apply(add_clades_to_gtdb_name)

In [20]:
merged.head()

,R202,R207,R214,VMGC_SGB,taxid_R214,VMGC_species,R202_with_clades,R207_with_clades,R214_with_clades
0,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB001,360001829,Lactobacillus iners,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Bacillota;c__Bacilli;o__Lactoba...
14,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB019,360001829,Lactobacillus iners,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Bacillota;c__Bacilli;o__Lactoba...
112,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB135,360001829,Lactobacillus iners,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Bacillota;c__Bacilli;o__Lactoba...
156,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB191,360001829,Lactobacillus iners,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Bacillota;c__Bacilli;o__Lactoba...
182,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Firmicutes;Bacilli;Lactobacillales;La...,Bacteria;Bacillota;Bacilli;Lactobacillales;Lac...,SGB219,360001829,Lactobacillus iners,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,d__Bacteria;p__Bacillota;c__Bacilli;o__Lactoba...


In [21]:
merged.to_csv('lineages_R202_R207_R214.csv')